In [30]:
import pandas as pd

In [31]:
data = pd.read_csv("D:\python-venv\Text\Imdb\data\IMDB Dataset.csv")

In [32]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [33]:
data['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [34]:
from bs4 import BeautifulSoup

In [35]:
def text_cleaner(text):
    #HTML to text format
    soup = BeautifulSoup(text)
    text = soup.get_text()
    return text

In [36]:
data['review'] = data['review'].apply(text_cleaner)

In [37]:
from sklearn.preprocessing import LabelEncoder

In [38]:
le = LabelEncoder()
y = le.fit_transform(data['sentiment'])

In [39]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [40]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['review'])

In [41]:
vocab_len = len(tokenizer.word_index) + 1

In [42]:
vocab_len

126510

In [43]:
encoded = tokenizer.texts_to_sequences(data['review'])

For finding the maximum of sentence len to set all sentences len with it

In [44]:
list_len = []
for seq in encoded:
    list_len.append(len(seq))

max_len = max(list_len)

In [45]:
max_len

2466

Importing pad sequnece for padding all sentences to the same size using tje max_len 

In [46]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [47]:
X = pad_sequences(encoded,maxlen=max_len,padding='post')

In [48]:
X

array([[  26,    4,    1, ...,    0,    0,    0],
       [   3,  392,  119, ...,    0,    0,    0],
       [   9,  189,   10, ...,    0,    0,    0],
       ...,
       [   9,  234,    3, ...,    0,    0,    0],
       [ 144,  165,    5, ...,    0,    0,    0],
       [  53,   26, 5889, ...,    0,    0,    0]])

In [49]:
X.shape

(50000, 2466)

In [50]:
from sklearn.model_selection import train_test_split

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [55]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding,Conv1D,MaxPool1D,Flatten,Dense,Dropout,Input
from tensorflow.keras.models import Sequential

In [56]:
reg = tf.keras.regularizers.l2(0.003)

In [63]:
model = Sequential()
model.add(Input(shape=(max_len,)))
model.add(Embedding(input_dim=vocab_len,output_dim=50,
                    embeddings_regularizer=reg))
model.add(Dropout(0.5))
model.add(Conv1D(filters=32,kernel_size=4,padding='same',activation='relu'))
model.add(Dropout(0.25))
model.add(MaxPool1D(pool_size=2))
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))

In [64]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 2466, 50)       │     6,325,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 2466, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 2466, 32)       │         6,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 2466, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 1233, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 39456)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │     2,525,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,857,245 (33.79 MB)

 Trainable params: 8,857,245 (33.79 MB)

 Non-trainable params: 0 (0.00 B)

In [65]:
#compile model
opt = tf.optimizers.Adam(learning_rate=0.001)
loss = tf.losses.BinaryCrossentropy()
metric = [tf.metrics.BinaryAccuracy()]
model.compile(optimizer=opt,loss=loss,metrics=metric)

In [66]:
model.fit(X_train,y_train,batch_size=50,epochs=10,validation_split=0.2,verbose=2)

Epoch 1/10
640/640 - 196s - 306ms/step - binary_accuracy: 0.7292 - loss: 0.9705 - val_binary_accuracy: 0.8550 - val_loss: 0.5500
Epoch 2/10
640/640 - 171s - 267ms/step - binary_accuracy: 0.8623 - loss: 0.5694 - val_binary_accuracy: 0.8824 - val_loss: 0.5245
Epoch 3/10
640/640 - 227s - 355ms/step - binary_accuracy: 0.8748 - loss: 0.5576 - val_binary_accuracy: 0.8845 - val_loss: 0.5572
Epoch 4/10
640/640 - 205s - 321ms/step - binary_accuracy: 0.8848 - loss: 0.5439 - val_binary_accuracy: 0.8926 - val_loss: 0.5231
Epoch 5/10
640/640 - 110s - 172ms/step - binary_accuracy: 0.8912 - loss: 0.5433 - val_binary_accuracy: 0.8923 - val_loss: 0.5351
Epoch 6/10
640/640 - 106s - 166ms/step - binary_accuracy: 0.8938 - loss: 0.5366 - val_binary_accuracy: 0.8924 - val_loss: 0.5273
Epoch 7/10
640/640 - 107s - 168ms/step - binary_accuracy: 0.8992 - loss: 0.5284 - val_binary_accuracy: 0.8936 - val_loss: 0.5350
Epoch 8/10
640/640 - 107s - 167ms/step - binary_accuracy: 0.9007 - loss: 0.5266 - val_binary_accu

In [67]:
model.evaluate(X_test,y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - binary_accuracy: 0.8746 - loss: 0.5800


[0.5799646973609924, 0.8745999932289124]